# Planejamento de rotas hospitalares

Este notebook reúne o fluxo completo de planejamento: leitura ou criação do cenário, diagnóstico dos dados, otimização, comparação experimental, visualização e geração de instruções operacionais. A proposta é que o processo possa ser acompanhado do início ao fim sem depender de um terminal.

Ao longo da execução, cada seção responde a uma pergunta diferente:

1. os dados são coerentes com a frota disponível?
2. como o algoritmo genético representa e avalia uma rota?
3. a busca está convergindo e a solução respeita as restrições?
4. o resultado é melhor que referências simples e estável entre sementes?
5. como transformar a solução em artefatos e instruções compreensíveis?

No VS Code ou Jupyter, selecione o kernel da `.venv` e execute as células em ordem. Depois de alterar o cenário ou algum parâmetro, execute novamente todas as células posteriores para evitar a comparação de resultados produzidos com configurações diferentes.


## 1. Preparação do ambiente

A primeira célula identifica automaticamente a raiz do repositório e instala o pacote em modo editável no kernel selecionado. Nesse modo, os módulos continuam apontando para o código em `src/hospital_routes`; portanto, uma alteração no projeto pode ser utilizada sem criar uma cópia separada do pacote.

Essa preparação também adiciona a pasta `src` ao caminho de importação da sessão. A célula pode ser executada novamente com segurança. Se os imports permanecerem carregando por muito tempo, confirme se o kernel escolhido pertence à `.venv` do projeto e reinicie a sessão antes de tentar outra vez.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT), '--quiet'])
sys.path.insert(0, str(ROOT / 'src'))
print(f'Projeto preparado em: {ROOT}')

## 2. Importações

As importações refletem a separação de responsabilidades adotada no projeto:

- `models` e `io` representam e carregam o problema;
- `genetic` executa a otimização;
- `baselines` fornece abordagens de referência;
- `visualization` produz mapa e gráficos;
- `reporting` prepara o contexto e gera os relatórios.

Ao final, o notebook informa qual interpretador está em uso e apenas confirma se uma chave do Gemini foi localizada. O valor da credencial nunca é exibido.


In [ ]:
import json
import random
import statistics
import time
from IPython.display import Markdown, clear_output, display
import matplotlib.pyplot as plt

from hospital_routes.baselines import brute_force, nearest_neighbor
from hospital_routes.genetic import GAConfig, GeneticOptimizer
from hospital_routes.io import load_problem, save_solution
from hospital_routes.models import Delivery, Depot, Problem, Vehicle
from hospital_routes.reporting import (GeminiReportGenerator, LocalReportGenerator, build_prompt, comparison_metrics, configured_gemini_key)
from hospital_routes.visualization import create_route_map, save_convergence_plot, save_route_map
print(f'Kernel: {sys.executable}')
print('Gemini configurado:', bool(configured_gemini_key()))

## 3. Configuração do experimento

Os valores desta seção concentram as escolhas que normalmente são alteradas durante os testes. Isso permite comparar cenários sem modificar o código do algoritmo.

| Parâmetro | Significado | Efeito esperado |
|---|---|---|
| `USAR_CENARIO_PADRAO` | escolhe entre o JSON versionado e um cenário sintético | `True` favorece reprodutibilidade; `False` permite variar o problema |
| `NUMERO_ENTREGAS` e `NUMERO_VEICULOS` | dimensões do cenário sintético | mais entregas ampliam rapidamente o espaço de busca |
| `CAPACIDADE_KG` e `AUTONOMIA_KM` | limites dos veículos sintéticos | valores baixos aumentam a chance de inviabilidade |
| `TAMANHO_POPULACAO` | quantidade de indivíduos por geração | populações maiores exploram mais alternativas, com maior custo computacional |
| `GERACOES` | limite de evolução | oferece mais oportunidades de melhoria, embora possa ocorrer parada antecipada |
| `TAXA_CROSSOVER` | probabilidade de recombinar dois pais | controla a combinação de diferentes ordens de visita |
| `TAXA_MUTACAO` | probabilidade de alterar um descendente | ajuda a preservar diversidade e escapar de ótimos locais |
| `ELITISMO` | melhores indivíduos preservados | zero permite testar sem elitismo; valores positivos evitam perder boas soluções |
| `SEMENTE` | estado inicial do gerador pseudoaleatório | permite repetir o mesmo experimento |

Quando `USAR_CENARIO_PADRAO = True`, quantidade de entregas, quantidade de veículos, capacidade e autonomia são lidas de `data/deliveries.json`; os quatro valores sintéticos correspondentes não são utilizados. Os parâmetros do algoritmo genético continuam válidos nos dois modos.


In [ ]:
USAR_CENARIO_PADRAO = True
NUMERO_ENTREGAS = 10
NUMERO_VEICULOS = 3
CAPACIDADE_KG = 55.0
AUTONOMIA_KM = 65.0

TAMANHO_POPULACAO = 120
GERACOES = 300
TAXA_CROSSOVER = 0.90
TAXA_MUTACAO = 0.20
ELITISMO = 4
SEMENTE = 42

## 4. Construção do cenário

O cenário é composto por um hospital, pelas entregas e pela frota. O hospital funciona como depósito: toda rota parte dele e retorna a ele, mas ele não é contado como entrega.

Cada entrega possui coordenadas em graus decimais, demanda total em quilogramas, tempo de serviço e prioridade. Neste projeto, prioridade `1` representa atendimento regular, `2` indica prioridade alta e `3` representa uma entrega crítica. A demanda descreve a carga total destinada ao ponto, e não o peso de uma unidade de medicamento.

No modo sintético, a mesma semente reproduz as mesmas posições, demandas e prioridades. A listagem exibida depois da célula resume o cenário usado e facilita a conferência das coordenadas antes da otimização.


In [ ]:
if USAR_CENARIO_PADRAO:
    problem = load_problem(ROOT / 'data' / 'deliveries.json')
else:
    rng = random.Random(SEMENTE)
    depot = Depot('Hospital Central', rng.uniform(-13.005, -12.94), rng.uniform(-38.53, -38.45))
    deliveries = tuple(
        Delivery(
            id=f'E{i + 1:02d}', name=f'Ponto {i + 1:02d}',
            latitude=rng.uniform(-13.05, -12.89), longitude=rng.uniform(-38.57, -38.33),
            demand_kg=round(rng.uniform(4, CAPACIDADE_KG * 0.30), 1),
            priority=rng.randint(1, 3), service_minutes=10,
        ) for i in range(NUMERO_ENTREGAS)
    )
    vehicles = tuple(Vehicle(f'VEIC-{i + 1:02d}', CAPACIDADE_KG, AUTONOMIA_KM) for i in range(NUMERO_VEICULOS))
    problem = Problem(depot, deliveries, vehicles)

print(f'Hospital: {problem.depot.latitude:.6f}, {problem.depot.longitude:.6f}')
print(f'Entregas: {len(problem.deliveries)} | Veículos: {len(problem.vehicles)}')
for delivery in problem.deliveries:
    print(f'{delivery.id}: ({delivery.latitude:.6f}, {delivery.longitude:.6f}) | {delivery.demand_kg:.1f} kg | prioridade {delivery.priority}')

### 4.1 Diagnóstico dos dados e viabilidade mínima

Antes de iniciar uma busca mais custosa, são verificadas duas condições necessárias:

- a soma das capacidades deve comportar a demanda total;
- a maior entrega deve caber em pelo menos um veículo.

Se uma dessas condições for falsa, o cenário já possui uma incompatibilidade evidente e deve ser revisto. Se ambas forem verdadeiras, ainda não há garantia de viabilidade: a divisão das cargas pode ser desfavorável e a autonomia depende da ordem e da distribuição espacial das visitas. Por isso, a confirmação definitiva ocorre somente após avaliar as rotas completas.


In [ ]:
total_demand = sum(item.demand_kg for item in problem.deliveries)
total_capacity = sum(vehicle.capacity_kg for vehicle in problem.vehicles)
largest_demand = max(item.demand_kg for item in problem.deliveries)
largest_capacity = max(vehicle.capacity_kg for vehicle in problem.vehicles)
diagnostics = {
    'demanda_total_kg': total_demand,
    'capacidade_total_kg': total_capacity,
    'capacidade_agregada_suficiente': total_demand <= total_capacity,
    'maior_entrega_atendivel': largest_demand <= largest_capacity,
}
diagnostics

## 5. Configuração do algoritmo genético

Cada indivíduo da população é uma permutação dos índices das entregas. Um decodificador percorre essa ordem e distribui as visitas entre os veículos, levando em conta a ocupação da carga, a distância projetada e possíveis violações. Essa estratégia preserva uma representação típica do TSP e permite ampliar o problema para múltiplos veículos.

A função de aptidão é minimizada e pode ser resumida por:

\[
f(x) = D + \alpha P + \beta E_c + \gamma E_a
\]

`D` representa a distância total; `P`, a distância acumulada até cada entrega ponderada por sua prioridade; `E_c` e `E_a`, os excessos de capacidade e autonomia. As penalidades elevadas tornam uma violação operacional mais relevante do que uma pequena redução de percurso.

A evolução utiliza seleção por torneio, crossover ordenado (OX1), mutações por troca ou inversão e elitismo configurável. O vizinho mais próximo é calculado antes da busca para estabelecer uma referência simples com a mesma função de aptidão.


In [ ]:
config = GAConfig(
    population_size=TAMANHO_POPULACAO, generations=GERACOES,
    crossover_rate=TAXA_CROSSOVER, mutation_rate=TAXA_MUTACAO,
    elite_size=ELITISMO, seed=SEMENTE,
)
optimizer = GeneticOptimizer(problem, config)
baseline = nearest_neighbor(optimizer)
print(f'Fitness do vizinho mais próximo: {baseline.fitness:.2f}')

## 6. Evolução e acompanhamento da convergência

Durante a evolução, a saída é atualizada a cada dez gerações. Os painéis usam escalas independentes para que cada comportamento permaneça legível:

- **melhor fitness:** acompanha a melhor alternativa da geração;
- **fitness médio:** resume a qualidade de toda a população.

Como o objetivo é minimizar, uma queda indica melhoria. Uma diferença ampla entre a média e o melhor valor sugere que ainda existe diversidade na população; a aproximação entre as curvas costuma indicar maior concentração em soluções semelhantes. A estabilização não prova que o ótimo global foi encontrado: ela mostra apenas que a configuração atual deixou de produzir melhorias relevantes.

Além do limite de gerações, o algoritmo interrompe a execução quando alcança o número configurado de gerações sem melhora. Assim, o total executado pode ser menor que `GERACOES`.


In [ ]:
solution = None
for stats, solution in optimizer.evolve():
    if stats.generation % 10 == 0 or stats.generation == GERACOES - 1:
        clear_output(wait=True)
        generations = [item.generation for item in optimizer.history]
        fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
        axes[0].plot(generations, [item.best for item in optimizer.history], color='#24966f')
        axes[0].set_title('Melhor fitness'); axes[0].set_ylabel('Fitness'); axes[0].grid(alpha=.25)
        axes[1].plot(generations, [item.mean for item in optimizer.history], color='#657f9f')
        axes[1].set_title('Fitness médio'); axes[1].set_xlabel('Geração'); axes[1].set_ylabel('Fitness'); axes[1].grid(alpha=.25)
        fig.tight_layout(); display(fig); plt.close(fig)
        print(f'Geração {stats.generation} | melhor: {stats.best:.2f} | média: {stats.mean:.2f}')

assert solution is not None
print(f'Concluído em {len(optimizer.history)} gerações.')

## 7. Análise da solução

Esta etapa separa métricas que respondem a perguntas diferentes. A distância total descreve o percurso, enquanto o fitness também incorpora prioridade e penalidades. Por esse motivo, a rota de menor distância nem sempre é a de menor fitness.

A solução é considerada viável somente quando nenhum veículo excede capacidade ou autonomia. Para cada rota são exibidos sequência de paradas, distância e carga; “não utilizado” apenas indica que o veículo ficou sem entregas, o que não constitui uma violação.

A variação percentual é calculada em relação ao vizinho mais próximo. Um valor positivo indica melhora do algoritmo genético; um valor negativo mostra que a referência obteve fitness menor naquela execução. A comparação é consistente porque as duas soluções usam a mesma função de avaliação.


In [ ]:
improvement = 100 * (baseline.fitness - solution.fitness) / baseline.fitness
print(f'Fitness final: {solution.fitness:.2f}')
print(f'Distância total: {solution.total_distance_km:.2f} km')
print(f'Solução viável: {solution.feasible}')
print(f'Redução frente ao vizinho mais próximo: {improvement:.2f}%')
for i, route in enumerate(solution.routes):
    metric = solution.metrics[i]
    stops = ' → '.join(problem.deliveries[j].id for j in route) or 'não utilizado'
    print(f'{problem.vehicles[i].id}: {stops} | {metric.distance_km:.2f} km | {metric.load_kg:.1f} kg')
comparison = comparison_metrics(solution, baseline)
comparison

## 8. Comparação experimental e robustez

Uma única execução não é suficiente para avaliar uma meta-heurística. A comparação foi dividida em duas partes complementares:

1. **instância reduzida:** as oito primeiras entregas são avaliadas por vizinho mais próximo, força bruta e algoritmo genético. A força bruta enumera `8! = 40.320` permutações e fornece o melhor resultado no espaço avaliado;
2. **cenário completo:** cinco sementes medem quanto o resultado varia quando a população inicial e as operações aleatórias mudam.

Todas as abordagens utilizam a mesma função de aptidão dentro de cada instância. Os tempos medidos com `perf_counter` servem apenas para comparar esta execução no equipamento atual; não representam garantia de desempenho em outro ambiente.


In [ ]:
small_problem = Problem(problem.depot, problem.deliveries[:8], problem.vehicles)
small_optimizer = GeneticOptimizer(
    small_problem, GAConfig(population_size=80, generations=150, stagnation_limit=50, seed=42)
)
times = {}
start = time.perf_counter(); nearest_small = nearest_neighbor(small_optimizer); times['vizinho'] = time.perf_counter() - start
start = time.perf_counter(); exact_small = brute_force(small_optimizer, max_deliveries=8); times['forca_bruta'] = time.perf_counter() - start
start = time.perf_counter(); genetic_small = small_optimizer.run(); times['genetico'] = time.perf_counter() - start
print('Instância reduzida (8 entregas)')
for name, item in [('Vizinho mais próximo', nearest_small), ('Força bruta', exact_small), ('Algoritmo genético', genetic_small)]:
    key = {'Vizinho mais próximo':'vizinho','Força bruta':'forca_bruta','Algoritmo genético':'genetico'}[name]
    print(f'{name}: fitness={item.fitness:.2f}; distância={item.total_distance_km:.2f} km; tempo={times[key]:.4f} s')
print('GA atingiu o fitness da força bruta:', abs(genetic_small.fitness - exact_small.fitness) < 1e-9)

In [ ]:
seed_results = []
for seed in (7, 21, 42, 84, 123):
    experiment = GeneticOptimizer(
        problem, GAConfig(population_size=80, generations=150, stagnation_limit=50, seed=seed)
    )
    reference = nearest_neighbor(experiment)
    start = time.perf_counter(); result = experiment.run(); elapsed = time.perf_counter() - start
    reduction = 100 * (reference.fitness - result.fitness) / reference.fitness
    seed_results.append((seed, result.fitness, result.total_distance_km, result.feasible, elapsed, reduction))
for row in seed_results:
    print(f'seed={row[0]} | fitness={row[1]:.2f} | distância={row[2]:.2f} km | viável={row[3]} | tempo={row[4]:.3f} s | redução={row[5]:.2f}%')
print(f'Fitness médio: {statistics.mean(r[1] for r in seed_results):.2f}')
print(f'Desvio-padrão do fitness: {statistics.pstdev(r[1] for r in seed_results):.2f}')
print(f'Redução média frente à referência: {statistics.mean(r[5] for r in seed_results):.2f}%')

### 8.1 Como interpretar o comparativo

Na instância reduzida, atingir o mesmo fitness da força bruta é uma evidência de qualidade para aquele recorte, e não uma prova de que o algoritmo genético sempre encontra o ótimo. No cenário completo, média e desvio-padrão mostram o comportamento conjunto das sementes: média menor indica melhor resultado global, enquanto desvio-padrão baixo sugere maior estabilidade.

Também é importante observar a coluna `viável`. Um fitness numericamente menor não deve ser aceito de forma isolada se houver excesso de carga ou autonomia. Por fim, o tempo de execução deve ser analisado junto com o tamanho da instância e a qualidade obtida; métodos exatos se tornam rapidamente impraticáveis conforme o número de entregas aumenta.


## 9. Geração dos entregáveis

Depois da análise, a solução é transformada em artefatos que podem ser inspecionados fora do notebook:

| Arquivo | Conteúdo |
|---|---|
| `solution.json` | rotas, métricas, fitness e viabilidade em formato estruturado |
| `routes_map.html` | mapa interativo com hospital, paradas e cores por veículo |
| `convergence.png` | painéis finais de melhor fitness e fitness médio |
| `daily_report.md` | instruções operacionais produzidas pelo Gemini ou pelo gerador local |
| `comparison.json` | métricas mensuráveis frente ao vizinho mais próximo |

Os arquivos são gravados em `outputs` e substituem versões anteriores com o mesmo nome. O JSON da solução permanece como fonte de verdade; o relatório textual é uma forma de comunicação derivada desses dados.


In [ ]:
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(exist_ok=True)
save_solution(OUTPUT / 'solution.json', problem, solution)
save_route_map(problem, solution, OUTPUT / 'routes_map.html')
save_convergence_plot(optimizer.history, OUTPUT / 'convergence.png')
try:
    report = GeminiReportGenerator().generate(problem, solution, comparison=comparison)
    report_source = 'Gemini'
except RuntimeError as error:
    report = LocalReportGenerator().generate(problem, solution, comparison=comparison)
    report_source = f'gerador local; Gemini indisponível: {error}'
(OUTPUT / 'daily_report.md').write_text(report, encoding='utf-8')
(OUTPUT / 'comparison.json').write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Origem do relatório: {report_source}')
print('Arquivos gerados:')
for name in ('solution.json', 'routes_map.html', 'convergence.png', 'daily_report.md', 'comparison.json'):
    print(' -', OUTPUT / name)

## 10. Mapa e relatório no notebook

O mapa permite conferir visualmente a distribuição das entregas e a ordem das paradas. As linhas ligam as coordenadas diretamente e representam distâncias geodésicas; elas não reproduzem ruas, trânsito ou sentido das vias.

Logo abaixo do mapa, o relatório operacional é exibido no próprio notebook. Se o Gemini estiver disponível, o texto será generativo; caso contrário, o gerador local apresentará as mesmas rotas de forma determinística, com base nos dados da solução.


In [ ]:
route_map = create_route_map(problem, solution)
display(route_map)
display(Markdown(report))


## 11. Integração com a LLM

A LLM participa somente depois que a otimização termina. Ela não escolhe veículos, não altera a ordem das visitas e não decide se uma rota é viável. Seu papel é transformar a solução estruturada em instruções, sínteses e respostas mais fáceis de consultar.

O prompt contém papel, tarefa, idioma, regras contra informações inventadas e o contexto em JSON. Também proíbe converter distância em tempo ou dinheiro sem dados observados. Esse envio direto da solução funciona como aterramento: a resposta deve permanecer vinculada aos valores calculados.

Não há RAG nesta versão porque não existe uma coleção de protocolos ou manuais a ser pesquisada. Caso esse acervo seja incorporado futuramente, seus documentos deverão ter fonte e versão identificáveis. Independentemente da técnica, qualquer saída generativa precisa de revisão humana antes de orientar uma operação real.


In [ ]:
prompt_preview = build_prompt(problem, solution, comparison=comparison)
print('Caracteres do prompt:', len(prompt_preview))
print('Instruções de controle presentes:', all(term in prompt_preview for term in ('Não invente', 'Não converta', 'Contexto estruturado')))
print('O contexto enviado contém apenas dados operacionais fictícios e a solução calculada.')

### 11.1 Relatório semanal e perguntas em linguagem natural

As chamadas adicionais ficam desativadas por padrão para que **Executar tudo** não dependa de cota, rede ou disponibilidade do provedor. Ao habilitá-las, o mesmo contexto é reutilizado para produzir uma análise semanal e responder a uma pergunta operacional.

Essas respostas complementam a apresentação dos resultados, mas não substituem `solution.json`. Se a API estiver indisponível ou recusar a requisição, a célula informa o problema sem interromper o restante do notebook. Nenhuma credencial é impressa ou armazenada nas saídas.


In [ ]:
EXECUTAR_TAREFAS_ADICIONAIS_LLM = False
if EXECUTAR_TAREFAS_ADICIONAIS_LLM:
    try:
        generator = GeminiReportGenerator()
        weekly_report = generator.generate(problem, solution, task='weekly', comparison=comparison)
        question = 'Quais veículos atendem entregas de prioridade crítica?'
        answer = generator.generate(
            problem, solution, task=f'Responda de forma objetiva: {question}', comparison=comparison
        )
        display(Markdown(weekly_report))
        display(Markdown('**Pergunta:** ' + question + '\n\n**Resposta:** ' + answer))
    except RuntimeError as error:
        print(f'Não foi possível executar as tarefas adicionais: {error}')
else:
    print('Chamadas adicionais desativadas. Altere a variável para demonstrar relatório semanal e perguntas.')


## 12. Síntese metodológica

Os resultados devem ser interpretados em três níveis:

1. **validade computacional:** testes verificam entidades, distâncias, operadores, restrições, reprodutibilidade e integração dos relatórios;
2. **qualidade experimental:** métodos de referência, força bruta reduzida e múltiplas sementes ajudam a avaliar qualidade e estabilidade, sem afirmar que o ótimo global foi encontrado no cenário completo;
3. **validade operacional:** depende da qualidade das coordenadas, demandas, prioridades e limites informados.

A solução oferece uma base reproduzível para experimentação e demonstra como integrar otimização evolutiva, visualização e geração textual fundamentada. Para uma aplicação operacional seriam necessários, entre outros elementos, uma matriz viária, tempos históricos, janelas de atendimento, custos observados, regras institucionais e governança dos dados.

Ao registrar os parâmetros, a semente e os artefatos de cada execução, torna-se possível explicar não apenas qual rota foi encontrada, mas também sob quais condições o resultado foi produzido.
